# SentinelOps — Day 01: LLM Internals, Tokenization & KV Cache Lab

Welcome to **Day 1** of SentinelOps! This notebook provides interactive experiments to understand the internal mechanics of transformers without requiring large GPU resources.

### Objectives:
1. **Experiment 1:** Tokenization Traps in Cloud & DevOps Data.
2. **Experiment 2:** Scaled Dot-Product Self-Attention in Pure NumPy.
3. **Experiment 3:** KV Cache Memory Footprint & Context Sizing.

## 1. Experiment 1: Tokenization Dynamics on Infrastructure Data
Observe how standard tokenizers break down natural language vs. technical DevOps identifiers (ARNs, IP addresses, stack traces).

In [ ]:
import re

try:
    import tiktoken
    enc = tiktoken.get_encoding('cl100k_base')
    tokenizer_name = 'tiktoken (cl100k_base - GPT-4)'
    tokenize = lambda text: [enc.decode([t]) for t in enc.encode(text)]
except ImportError:
    tokenizer_name = 'Standard Subword Splitter (Simulation)'
    tokenize = lambda text: re.findall(r'[A-Za-z]+|\d+|[:/_\-.,!?]|[\s]+', text)

print(f'Active Tokenizer: {tokenizer_name}\n')

samples = [
    ('English Query', 'Database connection pool is saturated and rejecting requests.'),
    ('Postgres Error', 'FATAL: 53300: remaining connection slots are reserved'),
    ('AWS ARN', 'arn:aws:rds:us-east-1:123456789012:db:prod-orders-cluster'),
    ('IP & Port', '10.128.4.15:5432'),
    ('K8s Signal', 'pod/payment-worker terminated with exit_code:137 (OOMKilled)')
]

print(f'{"Category":<18} | {"Chars":<6} | {"Tokens":<6} | {"Tokens Breakdown"}')
print('-' * 75)
for category, text in samples:
    tokens = tokenize(text)
    print(f'{category:<18} | {len(text):<6} | {len(tokens):<6} | {tokens[:8]}')


## 2. Experiment 2: Scaled Dot-Product Self-Attention
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
Let's compute the attention matrix from scratch using NumPy to see how tokens attend to each other.

In [ ]:
import numpy as np
import math

np.random.seed(42)
sequence = ['FATAL', '53300', 'connection_limit', 'rds_primary', 'orders_db']
seq_len = len(sequence)
d_model, d_k, d_v = 8, 4, 4

# Random initial embeddings and projection weights
X = np.random.randn(seq_len, d_model)
W_q = np.random.randn(d_model, d_k) * 0.5
W_k = np.random.randn(d_model, d_k) * 0.5
W_v = np.random.randn(d_model, d_v) * 0.5

# 1. Q, K, V projections
Q, K, V = X @ W_q, X @ W_k, X @ W_v

# 2. Scaled Dot-Product
scores = (Q @ K.T) / math.sqrt(d_k)

# 3. Softmax along row
exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
attention_matrix = exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# Display heatmap matrix
print('Self-Attention Weights Matrix:')
header = '          ' + ''.join([f'{t:>16}' for t in sequence])
print(header)
print('-' * len(header))
for i, row_tok in enumerate(sequence):
    row_str = f'{row_tok:<10}'
    for j in range(seq_len):
        row_str += f'{attention_matrix[i, j]:16.3f}'
    print(row_str)


## 3. Experiment 3: KV Cache Memory & Multi-Turn Sizing Calculator
Calculate how much RAM is required by the KV cache as incident investigation conversations grow.

In [ ]:
# Architecture specs (e.g. LLaMA-3-8B style model)
layers = 32
kv_heads = 8
head_dim = 128
precision_bytes = 2  # float16

# 2 (Key + Value) * layers * kv_heads * head_dim * bytes
bytes_per_tok = 2 * layers * kv_heads * head_dim * precision_bytes

phases = [
    ('Turn 1: Initial Ingested Alert', 500),
    ('Turn 3: Runbooks Retrieved', 2500),
    ('Turn 5: DB Investigation Logs', 6000),
    ('Turn 7: Network Traces Appended', 12000),
    ('Turn 10: Multi-Agent Synthesis', 25000),
    ('Peak Complex Incident', 64000)
]

print(f'KV Cache Memory per token: {bytes_per_tok} bytes ({bytes_per_tok / 1024:.2f} KB)\n')
print(f'{"Investigation Stage":<35} | {"Tokens":<8} | {"KV Cache RAM (MB)"}')
print('-' * 60)
for stage, tokens in phases:
    ram_mb = (tokens * bytes_per_tok) / (1024 * 1024)
    print(f'{stage:<35} | {tokens:<8} | {ram_mb:10.2f} MB')
